# Paper 13 neck on YOLO26n: fixed single-cycle feedback pyramid (backbone = proposed ADown + ECA)

| Part | Source |
|---|---|
| **Backbone (layers 0–10)** | **Proposed (fixed for the whole neck study):** ADown at layers 3, 5, 7 + ECA after layers 6 (k = 3) and 8 (k = 5) |
| **Neck** | **Paper 13 (RFP-YOLO26):** the fixed single-cycle **feedback pyramid**: top-down → bottom-up → a **second top-down** refinement pass; the P3 and P4 detection inputs come from the second top-down pass |
| Head (Detect) | YOLO26n base, unchanged (one-to-one, NMS-free); it moves from layer 23 to layer 29 |

**Only the topology is taken from Paper 13.** Every fusion block stays a YOLO26n `C3k2` and every down-sampling stays a YOLO26n `Conv`:
- SPDConv (Paper 13's down-sampling) was rejected in the backbone study (3.68 FPS on the Pi 5),
- C3k2_Faster_EMA gave no gain in Paper 13's own ablation (mAP@50:95 −0.3) and EMA is slow on the Pi 5,
- SimAM is a head-side module (tested in the head study).
So this notebook measures the effect of the feedback topology alone.

The reference row of the neck table is the **proposed backbone with the YOLO26n neck** (`final_backbone.ipynb`); it is not trained here.
Same training settings as every other row (100 epochs, seed 0, MuSGD, imgsz 640, batch 16).

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**, keep the notebook **Private**. Then **Run All**.

## 1. Setup

In [1]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)

import time
NOTEBOOK_START = time.time()  # used by the Kaggle 12 h guard in the training cell

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
ultralytics 8.4.137 | torch 2.11.0+cu130 | torchvision 0.26.0+cu130
GPU: Tesla T4


## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [3]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to digitalaccess-2(copy)-4 in yolo26:: 100%|██████████| 17613/17613 [00:01<00:00, 11612.73it/s]


train : 7704 images
valid : 550 images
test  : 550 images
nc = 27
✅ Dataset prepared!


## 3. Backbone (fixed): ADown (3, 5, 7) + ECA (6, 8)

The same backbone as `final_backbone.ipynb`. ADown is set in the YAML; ECA is added by the trainer after `yolo26n.pt` is loaded.
The YAML below is the reference model (YOLO26n neck); the feedback model in section 4 uses the same backbone lines.

In [4]:
%%writefile yolo26n-final-backbone.yaml
# YOLO26n with the proposed backbone: stride-2 Conv at layers 3, 5, 7 -> ADown (ECA after 6, 8 is added by the trainer).
# Neck and head = YOLO26n base.
nc: 80 # overridden by the dataset
end2end: True
reg_max: 1
scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, ADown, [256]] # 3-P3/8   (Conv -> ADown)
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, ADown, [512]] # 5-P4/16  (Conv -> ADown)
  - [-1, 2, C3k2, [512, True]] # 6  (+ ECA k=3)
  - [-1, 1, ADown, [1024]] # 7-P5/32 (Conv -> ADown)
  - [-1, 2, C3k2, [1024, True]] # 8  (+ ECA k=5)
  - [-1, 1, SPPF, [1024, 5, 3, True]] # 9
  - [-1, 2, C2PSA, [1024]] # 10

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 2, C3k2, [512, True]] # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 2, C3k2, [256, True]] # 16 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]] # cat head P4
  - [-1, 2, C3k2, [512, True]] # 19 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] # cat head P5
  - [-1, 1, C3k2, [1024, True, 0.5, True]] # 22 (P5/32-large)

  - [[16, 19, 22], 1, Detect, [nc]] # Detect(P3, P4, P5)

Writing yolo26n-final-backbone.yaml


In [5]:
%%writefile eca_backbone.py
"""Proposed backbone for YOLO26n: ECA after the C3k2 at layer 6 (P4, k=3) and layer 8 (P5, k=5). ADown (3, 5, 7) is in the YAML."""
import torch.nn as nn

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.block import C3k2
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

ECA_LAYERS = {6: 3, 8: 5}  # layer index: ECA kernel size


class ECA(nn.Module):
    """Efficient Channel Attention: global avg pool -> conv across channels (kernel k) -> sigmoid -> scale. 1-D conv as 4-D Conv2d."""

    def __init__(self, k=3):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv2d(1, 1, (k, 1), padding=(k // 2, 0), bias=False)
        self.act = nn.Sigmoid()

    def forward(self, x):
        y = self.pool(x).transpose(1, 2)  # (B, C, 1, 1) -> (B, 1, C, 1)
        y = self.act(self.conv(y)).transpose(1, 2)  # back to (B, C, 1, 1)
        return x * y


class C3k2ECA(nn.Module):
    """A pretrained C3k2 block followed by ECA."""

    def __init__(self, block, k=3):
        super().__init__()
        self.block = block
        self.eca = ECA(k)

    def forward(self, x):
        return self.eca(self.block(x))


def add_eca(model, layers=ECA_LAYERS):
    """Wrap the C3k2 at the given layers with ECA (the C3k2 keeps its weights). Returns the model."""
    for i, k in layers.items():
        old = model.model[i]
        assert isinstance(old, C3k2), f"layer {i} is {type(old).__name__}, not C3k2"
        new = C3k2ECA(old, k).to(next(old.parameters()).device)
        new.i, new.f, new.type = old.i, old.f, "C3k2ECA"  # keep the routing attributes used by the model's forward
        new.np = sum(p.numel() for p in new.parameters())
        model.model[i] = new
    return model


class ECATrainer(DetectionTrainer):
    """Builds the model from the ADown YAML, loads the pretrained weights, then adds ECA after the C3k2 at layers 6 and 8."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_eca(model)
        if RANK in {-1, 0}:
            print(f"Proposed backbone: ADown at layers 3, 5, 7 + ECA after layers {list(ECA_LAYERS)} (k = {list(ECA_LAYERS.values())})")
        if verbose and RANK in {-1, 0}:
            model.info()
        return model

Writing eca_backbone.py


In [6]:
%%writefile fix_eca_ncnn.py
"""Re-export ECA-style models to NCNN without the Segmentation fault (papers 10, 17, 19, 49).

Why: ECA blocks do pool -> transpose(1,2) -> Conv2d(1,1,(k,1)) -> transpose. PNNX turns the first transpose into a
no-op Permute, so the 1-channel conv receives C channels and NCNN crashes (Segmentation fault).

Fix: the cross-channel 1-D conv is rewritten as an exactly equivalent 1x1 Conv2d(C, C) whose weight is a banded
matrix built from the trained kernel. Same math, same outputs, no transposes -> NCNN-safe.

Run inside the unzipped bench folder (needs bb_*.pt and the *_backbone.py file):
    python fix_eca_ncnn.py            # check + export every bb_*.pt that contains ECA (replaces <name>_ncnn_model)
    python fix_eca_ncnn.py --check    # only check equivalence, no export
Models without ECA blocks are left untouched.
"""
import glob
import importlib
import shutil
import sys
from pathlib import Path

import torch
import torch.nn as nn

sys.path.insert(0, ".")
for f in glob.glob("*_backbone.py"):  # torch needs these to unpickle the custom modules
    importlib.import_module(Path(f).stem)

from ultralytics import YOLO  # noqa: E402


class ECAConv1x1(nn.Module):
    """ECA with the cross-channel conv as a banded 1x1 Conv2d(C, C). scale=True -> x * att, False -> att only."""

    def __init__(self, conv1d: nn.Conv2d, c: int, scale: bool):
        super().__init__()
        w = conv1d.weight.detach().view(-1).float()  # (k,)
        k, p = w.numel(), conv1d.padding[0]
        band = torch.zeros(c, c)
        for o in range(c):  # out[o] = sum_j w[j] * in[o + j - p]  (zero padding)
            for j in range(k):
                i = o + j - p
                if 0 <= i < c:
                    band[o, i] = w[j]
        self.scale = scale
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv2d(c, c, 1, bias=False)
        self.conv.weight.data.copy_(band.view(c, c, 1, 1))
        self.act = nn.Sigmoid()

    def forward(self, x):
        y = self.act(self.conv(self.pool(x)))
        return x * y if self.scale else y


def is_eca(m):
    """A block whose .conv is Conv2d(1, 1, (k, 1)) with k > 1 - the transpose-based ECA pattern."""
    c = getattr(m, "conv", None)
    return (isinstance(c, nn.Conv2d) and c.in_channels == 1 and c.out_channels == 1
            and c.kernel_size[1] == 1 and c.kernel_size[0] > 1 and not isinstance(m, ECAConv1x1))


@torch.no_grad()
def patch(net):
    """Replace every ECA block. Returns (count, max abs difference old vs new)."""
    names = [n for n, m in net.named_modules() if is_eca(m)]
    if not names:
        return 0, 0.0
    chans = {}

    def hook(name):
        def f(m, inp, out):  # must return None, otherwise it would replace the block's output
            chans.setdefault(name, inp[0].shape[1])
        return f

    hooks = [net.get_submodule(n).register_forward_hook(hook(n)) for n in names]
    net(torch.zeros(1, 3, 640, 640))
    for h in hooks:
        h.remove()

    diff = 0.0
    for n in names:
        old, c = net.get_submodule(n), chans[n]
        x = torch.randn(2, c, 20, 20)
        ref = old(x)
        new = None
        for scale in (True, False):  # find which variant this block is (returns x*att or att only)
            cand = ECAConv1x1(old.conv, c, scale).eval()
            out = cand(x)
            if out.shape == ref.shape and (out - ref).abs().max().item() < 1e-4:
                new = cand
                diff = max(diff, (out - ref).abs().max().item())
                break
        if new is None:
            sys.exit(f"Block {n} ({type(old).__name__}) did not match - not exporting.")
        parent, _, attr = n.rpartition(".")
        setattr(net.get_submodule(parent) if parent else net, attr, new)
    return len(names), diff


def main():
    pts = sorted(p for p in glob.glob("bb_*.pt") if not p.endswith("_fixed.pt"))
    if not pts:
        sys.exit("No bb_*.pt found. Run this inside the unzipped bench folder.")
    for pt in pts:
        yolo = YOLO(pt, task="detect")
        net = yolo.model.float().eval()
        n, diff = patch(net)
        if n == 0:
            print(f"{pt}: no ECA blocks - left as is.")
            continue
        print(f"{pt}: replaced {n} ECA blocks, max block output difference = {diff:.2e}  -> Check OK")
        if "--check" in sys.argv:
            continue

        name = Path(pt).stem
        tmp = Path(f"{name}_fixed.pt")
        yolo.save(tmp)
        exported = Path(YOLO(str(tmp), task="detect").export(format="ncnn", imgsz=640))
        out_dir = Path(f"{name}_ncnn_model")
        if out_dir.exists():
            shutil.rmtree(out_dir)
        shutil.move(str(exported), out_dir)
        tmp.unlink(missing_ok=True)
        Path(f"{name}_fixed.torchscript").unlink(missing_ok=True)
        print(f"{pt}: Done -> {out_dir}/")
    print("All done. Now run: python pi5_bench.py")


if __name__ == "__main__":
    main()

Writing fix_eca_ncnn.py


## 4. Paper 13 neck: fixed single-cycle feedback pyramid

Paper 13, Section 3.4, Equations (18)–(21), with `F` = YOLO26n C3k2, `D` = YOLO26n 3×3 stride-2 Conv, `U` = 2× nearest upsampling:

| Pass | Paper 13 | Layer here | Same as YOLO26n? |
|---|---|---|---|
| 1st top-down | T4 = F[Concat(U(B5), B4)] | 11–13 | yes |
| | T3 = F[Concat(U(T4), B3)] | 14–16 | yes |
| bottom-up feedback | F4 = F[Concat(D(T3), T4)] | 17–19 | yes |
| | F5 = F[Concat(D(F4), B5)] | 20–22 | yes (P5 output) |
| **2nd top-down (new)** | **R4 = F[Concat(U(F5), F4)]** | **23–25** | new |
| | **R3 = F[Concat(U(R4), T3)]** | **26–28** | new |
| Detect | (R3, R4, F5) | **29** | Detect(P3 = 28, P4 = 25, P5 = 22) |

Layers 0–22 are exactly the reference model, so the new part is the second top-down pass (2 upsamples, 2 concats, 2 C3k2).
The two new C3k2 do not share weights with the first top-down pass (as in Paper 13).

**Pretrained start.** Layers 0–22 load `yolo26n.pt` as usual. Detect moved from index 23 to 29, so its pretrained weights are copied over by hand.
R4 (layer 25) has exactly the same input and output shape as T4 (layer 13), so it starts from the pretrained layer-13 weights.
R3 (layer 28) has a different input width than layer 16, so it starts from scratch.

**Cost.** Paper 13 reports the feedback pyramid as the single largest accuracy gain but also the largest cost; the Pi 5 FPS decides.

In [7]:
%%writefile yolo26n-final-backbone-feedback.yaml
# YOLO26n, proposed backbone (ADown at layers 3, 5, 7; ECA after 6, 8 is added by the trainer)
# + Paper 13 fixed single-cycle feedback pyramid: top-down -> bottom-up -> second top-down. Head = YOLO26n Detect.
nc: 80 # overridden by the dataset
end2end: True
reg_max: 1
scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, ADown, [256]] # 3-P3/8   (Conv -> ADown)
  - [-1, 2, C3k2, [512, False, 0.25]] # 4 = B3
  - [-1, 1, ADown, [512]] # 5-P4/16  (Conv -> ADown)
  - [-1, 2, C3k2, [512, True]] # 6 = B4 (+ ECA k=3)
  - [-1, 1, ADown, [1024]] # 7-P5/32 (Conv -> ADown)
  - [-1, 2, C3k2, [1024, True]] # 8  (+ ECA k=5)
  - [-1, 1, SPPF, [1024, 5, 3, True]] # 9
  - [-1, 2, C2PSA, [1024]] # 10 = B5

head:
  # 1st top-down pass
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, True]] # 13 = T4
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, True]] # 16 = T3

  # bottom-up feedback pass
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, True]] # 19 = F4
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 1, C3k2, [1024, True, 0.5, True]] # 22 = F5 (P5/32-large)

  # 2nd top-down refinement pass (Paper 13, Eq. 20)
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 19], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, True]] # 25 = R4 (P4/16-medium)
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 16], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, True]] # 28 = R3 (P3/8-small)

  - [[28, 25, 22], 1, Detect, [nc]] # 29 Detect(P3, P4, P5)

Writing yolo26n-final-backbone-feedback.yaml


In [8]:
%%writefile feedback_neck.py
"""Paper 13 (RFP-YOLO26) neck for YOLO26n: fixed single-cycle feedback pyramid (top-down -> bottom-up -> second top-down),
built from the YAML yolo26n-final-backbone-feedback.yaml. Backbone = proposed ADown (3, 5, 7) + ECA (6, 8)."""
import torch

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

from eca_backbone import ECA_LAYERS, add_eca

# (layer in yolo26n.pt) -> (layer in the feedback model) for the weights that changed index or are copied to a new layer
COPY = {23: 29,  # Detect moved from 23 to 29
        13: 25}  # R4 has the same shapes as T4: start from the pretrained T4 weights


@torch.no_grad()
def copy_moved_weights(model, weights):
    """Copy pretrained tensors of the layers in COPY to their new index, where the shapes match. Returns the number of tensors copied."""
    src = weights.float().state_dict()
    dst = model.state_dict()
    n = 0
    for k, v in src.items():
        parts = k.split(".")
        if parts[0] != "model" or int(parts[1]) not in COPY:
            continue
        nk = ".".join(["model", str(COPY[int(parts[1])])] + parts[2:])
        if nk in dst and dst[nk].shape == v.shape:
            dst[nk].copy_(v)
            n += 1
    return n


class FeedbackNeckTrainer(DetectionTrainer):
    """Builds the feedback YAML, loads the pretrained weights (layers 0-22 by name, Detect and R4 by hand), then adds ECA (layers 6, 8)."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
            n = copy_moved_weights(model, weights)
            if RANK in {-1, 0}:
                print(f"Paper 13 feedback neck: {n} pretrained tensors copied to the moved/new layers {sorted(COPY.values())}")
        add_eca(model)
        if RANK in {-1, 0}:
            print(f"Backbone: ADown 3, 5, 7 + ECA after {list(ECA_LAYERS)} | Neck (Paper 13): feedback pyramid, Detect at layer 29")
        if verbose and RANK in {-1, 0}:
            model.info()
        return model

Writing feedback_neck.py


In [9]:
import importlib, eca_backbone, feedback_neck
importlib.reload(eca_backbone); importlib.reload(feedback_neck)
from eca_backbone import add_eca
from feedback_neck import FeedbackNeckTrainer, copy_moved_weights
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

REF_YAML = str(WORK / "yolo26n-final-backbone.yaml")             # proposed backbone + YOLO26n neck (reference row)
MODEL_YAML = str(WORK / "yolo26n-final-backbone-feedback.yaml")  # proposed backbone + Paper 13 feedback neck (trained here)

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

# structure check (untrained): base vs proposed backbone vs proposed backbone + feedback neck
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False).eval()
prop = add_eca(DetectionModel(yaml_model_load(REF_YAML), nc=len(names), verbose=False)).eval()
neck = DetectionModel(yaml_model_load(MODEL_YAML), nc=len(names), verbose=False)
print("tensors that would be copied from a YOLO26n model:", copy_moved_weights(neck, base))
neck = add_eca(neck).eval()
with torch.no_grad():
    _ = neck(torch.zeros(1, 3, 640, 640))  # full forward check
print("backbone layers 1, 3, 5, 6, 7, 8:", [type(neck.model[i]).__name__ for i in (1, 3, 5, 6, 7, 8)])
print("2nd top-down layers 25, 28:      ", [type(neck.model[i]).__name__ for i in (25, 28)], "| layer 29:", type(neck.model[29]).__name__, "inputs", neck.model[29].f)
print("strides:", neck.stride.tolist())
for tag, m in (("base YOLO26n", base), ("Proposed backbone", prop), ("+ feedback neck", neck)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:18s} Params {pm:.3f} M | GFLOPs {gf:.2f}")
print("expected: + feedback neck higher than the proposed backbone (~2.03 M | ~4.49 GFLOPs): two extra C3k2")

Overriding model.yaml nc=80 with nc=27
Overriding model.yaml nc=80 with nc=27
Overriding model.yaml nc=80 with nc=27
tensors that would be copied from a YOLO26n model: 294
backbone layers 1, 3, 5, 6, 7, 8: ['Conv', 'ADown', 'ADown', 'C3k2ECA', 'ADown', 'C3k2ECA']
2nd top-down layers 25, 28:       ['C3k2', 'C3k2'] | layer 29: Detect inputs [28, 25, 22]
strides: [8.0, 16.0, 32.0]
base YOLO26n       Params 2.380 M | GFLOPs 5.34
Proposed backbone  Params 2.034 M | GFLOPs 4.49
+ feedback neck    Params 2.183 M | GFLOPs 5.25
expected: + feedback neck higher than the proposed backbone (~2.03 M | ~4.49 GFLOPs): two extra C3k2


## 5. Training

Only the proposed backbone + feedback-pyramid neck model is trained (about 1.5–2.5 h on a T4).

In [10]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name)
VARIANTS = [
    ("Proposed Backbone + Feedback Neck (Paper 13)", "nk_feedback"),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

# Kaggle stops a session after 12 hours. Training stops after the epoch that passes TRAIN_STOP_HOURS (counted from the
# setup cell), so the test evaluation, NCNN export and zip still finish in time. best.pt is the best epoch reached so far.
TRAIN_STOP_HOURS = 10.5

def stop_before_kaggle_limit(trainer):
    hours = (time.time() - NOTEBOOK_START) / 3600
    if hours > TRAIN_STOP_HOURS and not trainer.stop:
        trainer.stop = True
        print(f"\n⏱ {hours:.1f} h since the notebook started: training stops after epoch {trainer.epoch + 1} (Kaggle 12 h limit).")

for label, run in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO(MODEL_YAML)
    model.add_callback("on_fit_epoch_end", stop_before_kaggle_limit)
    model.train(trainer=FeedbackNeckTrainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    n_ep = len(pd.read_csv(Path(PROJECT) / f"{run}_s{SEED}" / "results.csv"))
    print(f"{run}: trained {n_ep} of {EPOCHS} epochs" + ("" if n_ep >= EPOCHS else " (stopped early: patience or the 12 h guard)"))
    del model; gc.collect(); torch.cuda.empty_cache()


========== Proposed Backbone + Feedback Neck (Paper 13) (nk_feedback) ==========
New https://pypi.org/project/ultralytics/8.4.202 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.137 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/digitalaccess/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, ko

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


      2/100      3.72G      1.353      3.945    0.01827         14        640: 100% ━━━━━━━━━━━━ 482/482 4.2it/s 1:54
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.0it/s 2.6s
                   all        550        557      0.695      0.737      0.809      0.458

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/100      3.73G      1.414        1.9    0.01914         12        640: 100% ━━━━━━━━━━━━ 482/482 4.4it/s 1:51
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.0it/s 2.6s
                   all        550        557        0.8      0.803       0.89      0.525

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/100      3.73G      1.423      1.469    0.01922          7        640: 100% ━━━━━━━━━━━━ 482/482 4.4it/s 1:50
                 Class     Images  Instances      Box(P 

## 6. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the fused inference model, the same way as every other row.

In [11]:
import eca_backbone, feedback_neck  # must be imported before loading the checkpoint

rows = []
for label, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

Ultralytics 8.4.137 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n-final-backbone-feedback summary (fused): 153 layers, 2,183,245 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1134.5±258.6 MB/s, size: 37.7 KB)
val: Scanning /kaggle/working/digitalaccess/test/labels... 550 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 550/550 1.3Kit/s 0.4s
val: New cache created: /kaggle/working/digitalaccess/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 12.7it/s 2.8s
                   all        550        552      0.957      0.921      0.944      0.581
Speed: 0.5ms preprocess, 3.0ms inference, 0.0ms loss, 0.1ms postprocess per image


,Variant,Params (M),GFLOPs,Precision,Recall,F1-score,mAP@50
0,Proposed Backbone + Feedback Neck (Paper 13),2.18,5.25,0.957,0.921,0.938,0.944


## 7. Package for Raspberry Pi 5 and Jetson

Creates `paper13_neck_edge_bench/` with:

| Item | Used on |
|---|---|
| `nk_feedback_ncnn_model/` (NCNN model, imgsz 640, ECA as 1×1 conv) | Raspberry Pi 5 |
| `nk_feedback.pt` + `eca_backbone.py` + `feedback_neck.py` | Jetson |
| `bench_images/` (the same 50 real test images, seed 0) | both |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

Before the NCNN export, the 2 ECA blocks of the backbone are rewritten as exactly equivalent banded 1×1 convs (`fix_eca_ncnn.py`),
because the transpose-based ECA crashes NCNN. The `.pt` file for the Jetson keeps the original ECA.

In [12]:
import eca_backbone, feedback_neck  # needed to load the checkpoint
from fix_eca_ncnn import patch  # ECA -> banded 1x1 conv, NCNN-safe

EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper13_neck_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("eca_backbone.py", EDGE / "eca_backbone.py")
shutil.copy("feedback_neck.py", EDGE / "feedback_neck.py")

# the same 50 real test images as the earlier packages (same seed and selection)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    # NCNN-safe copy: ECA -> equivalent banded 1x1 conv (checked block by block), then export
    yolo = YOLO(str(w), task="detect")
    n, diff = patch(yolo.model.float().eval())
    print(f"{run}: replaced {n} ECA blocks for NCNN, max block output difference = {diff:.2e}")
    assert n == 2, "expected 2 ECA blocks (layers 6 and 8)"
    tmp = WORK / f"{run}_ncnn_safe.pt"
    yolo.save(tmp)
    p = YOLO(str(tmp), task="detect").export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
    shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
    print(f"{run:12s} -> {run}.pt, {run}_ncnn_model/")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import eca_backbone, feedback_neck  # needed to load the checkpoint
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
print("\nEdge package ready. Run section 9 to zip everything.")

bench_images: 50 real test images
nk_feedback: replaced 2 ECA blocks for NCNN, max block output difference = 0.00e+00
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.137 🚀 Python-3.13.16 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ This export format does not support end2end models, disabling the end2end branch.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n-final-backbone-feedback summary (fused): 177 layers, 2,389,890 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from '/kaggle/working/nk_feedback_ncnn_safe.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 31, 8400) (5.0 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 1 package in 201ms
 Downloaded ncnn
Prepared 1 package in 336ms
Installed 1 pack

pnnxparam = /kaggle/working/nk_feedback_ncnn_safe_ncnn_model/model.pnnx.param
pnnxbin = /kaggle/working/nk_feedback_ncnn_safe_ncnn_model/model.pnnx.bin
pnnxpy = /kaggle/working/nk_feedback_ncnn_safe_ncnn_model/model_pnnx.py
pnnxonnx = /kaggle/working/nk_feedback_ncnn_safe_ncnn_model/model.pnnx.onnx
ncnnparam = /kaggle/working/nk_feedback_ncnn_safe_ncnn_model/model.ncnn.param
ncnnbin = /kaggle/working/nk_feedback_ncnn_safe_ncnn_model/model.ncnn.bin
ncnnpy = /kaggle/working/nk_feedback_ncnn_safe_ncnn_model/model_ncnn.py
fp16 = 0
optlevel = 2
device = cpu
inputshape = [1,3,640,640]f32
inputshape2 = 
input = 
input2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0
inline module = eca_backbone.C3k2ECA
inline module = fix_eca_ncnn.ECAConv1x1
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.ADown
inline module = ultralytics.nn.modules.block.Attention
inline module = ultralytics.nn.

NCNN: export success ✅ 7.5s, saved as '/kaggle/working/nk_feedback_ncnn_safe_ncnn_model' (8.9 MB)

Export complete (8.1s)
Results saved to /kaggle/working/nk_feedback_ncnn_safe_ncnn_model
Predict:         yolo predict task=detect model=/kaggle/working/nk_feedback_ncnn_safe_ncnn_model imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/nk_feedback_ncnn_safe_ncnn_model imgsz=640 data=/kaggle/working/digitalaccess/data.yaml  
Visualize:       https://netron.app
nk_feedback  -> nk_feedback.pt, nk_feedback_ncnn_model/

Edge package ready. Run section 9 to zip everything.


### How to run on the Raspberry Pi 5

Copy `paper13_neck_all_results.zip` to the Pi and run the bench script:
```bash
~/fps/bench_paper.sh paper13_neck_all_results.zip
```

## 8. Neck table row

The reference row is the **proposed backbone with the YOLO26n neck** from `final_backbone.ipynb`. Type its values once they are known
(and the FPS of this model from the Pi 5), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 6 first in the same session (it fills `METRICS`).

In [13]:
# reference row: proposed backbone + YOLO26n neck (fill in from final_backbone.ipynb)
BASE_ROW = {"Variant": "Proposed Backbone + YOLO26n Neck", "Params (M)": 2.03, "GFLOPs": 4.49,
            "Precision": None, "Recall": None, "F1-score": None, "mAP@50": None,
            "FPS (Rasp-5)": None, "FPS (Jetson)": None}

FPS_PI5    = {"nk_feedback": None}
FPS_JETSON = {"nk_feedback": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


Variant,Params (M),GFLOPs,Precision,Recall,F1-score,mAP@50,FPS (Rasp-5),FPS (Jetson)
Proposed Backbone + YOLO26n Neck,2.030000,4.490000,--,--,--,--,--,--
Proposed Backbone + Feedback Neck (Paper 13),2.180000,5.250000,0.957000,0.921000,0.938000,0.944000,--,--


## 9. Zip all results

Run this last. It creates **`paper13_neck_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper13_neck_edge_bench/` | NCNN + `.pt` model, module files, 50 bench images, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `best.pt`, renamed `nk_feedback_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |
| `yolo26n-final-backbone-feedback.yaml`, `eca_backbone.py`, `feedback_neck.py` | the model definition |

In [14]:
import zipfile

ALL_ZIP = WORK / "paper13_neck_all_results.zip"
folders = [WORK / "paper13_neck_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")
    if Path(MODEL_YAML).exists():
        z.write(MODEL_YAML, Path(MODEL_YAML).name)
    z.write("eca_backbone.py", "eca_backbone.py")
    z.write("feedback_neck.py", "feedback_neck.py")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")

paper13_neck_edge_bench/       60 files
runs/                          21 files
weights/nk_feedback_best.pt

Download paper13_neck_all_results.zip (33.7 MB) from the Output tab.
